In [1]:
import torch
import numpy as np

In [3]:
vocab_size = 10000
embed_size = 512
seq_length = 20
batch_size = 32

# Create random input data (token IDs)
input_data = torch.randint(0, vocab_size, (batch_size, seq_length))
print(input_data.shape)

torch.Size([32, 20])


In [ ]:
w_embed = torch.rand(vocab_size, embed_size)
embedded_input = w_embed[input_data]

print(embedded_input.shape) # should be (32, 20, 512)

torch.Size([32, 20, 512])


In [5]:
# now we need query, key and value matrices
q_matrix = torch.rand(embed_size, embed_size)
k_matrix = torch.rand(embed_size, embed_size)
v_matrix = torch.rand(embed_size, embed_size)

Q = embedded_input @ q_matrix
K = embedded_input @ k_matrix
V = embedded_input @ v_matrix


In [7]:
Q.shape, K.shape, V.shape

(torch.Size([32, 20, 512]),
 torch.Size([32, 20, 512]),
 torch.Size([32, 20, 512]))

In [10]:
attention_score = Q @ K.transpose(-2, -1) / torch.sqrt(torch.tensor(embed_size))
print(attention_score.shape) # should be (32, 20, 20)

torch.Size([32, 20, 20])


In [11]:
attention_norm = torch.softmax(attention_score, dim=-1)
print(attention_norm.shape) # should be (32, 20, 20)

torch.Size([32, 20, 20])


In [12]:
out = attention_norm @ V
print(out.shape) # should be (32, 20, 512)

torch.Size([32, 20, 512])


In [14]:
#that is the whole attention layer simple  now after the attention layer we can add a feed forward network and then we can stack multiple layers of attention and feed forward network to create a transformer model.

x1 = embedded_input + out  # add residual connection

# we know the formula of feed forward network is FFN(x) = max(0, xW1 + b1)W2 + b2
w1 = torch.rand(embed_size, embed_size)
b1 = torch.rand(embed_size)
w2 = torch.rand(embed_size, embed_size)
b2 = torch.rand(embed_size)

ep = 1e-5
mean = x1.mean(dim=0, keepdim=True)
std = x1.std(dim=0, keepdim=True) + ep
gamma1 = torch.rand(embed_size)
beta1 = torch.rand(embed_size)

x1_norm = (x1 - mean) / std * gamma1 + beta1


In [15]:
hidden_size = embedded_input * 4 # usually the hidden size is 4 times the embed size

ff_hidden = torch.relu(x1_norm @ w1 + b1)
relu_out = torch.max(ff_hidden, torch.zeros_like(ff_hidden)) # relu activation
ff_out = relu_out @ w2 + b2

x2 = x1 + ff_out # add residual connection
mean2 = x2.mean(dim=0, keepdim=True)
std2 = x2.std(dim=0, keepdim=True) + ep
gamma2 = torch.rand(embed_size)
beta2 = torch.rand(embed_size)

final_encoder_output = (x2 - mean2) / std2 * gamma2 + beta2
print(final_encoder_output.shape) # should be (32, 20, 512)

torch.Size([32, 20, 512])
